# Video-to-3D Room Reconstruction — Person A

Reproducible Kaggle pipeline for video frame processing, VGGT/GLOMAP pose estimation, COLMAP-compatible sparse reconstruction, 3D Gaussian Splatting training, held-out evaluation, scene export, and backend handoff. Every stage is implemented as an independent command-line process with `--project_dir` support.

**Environment:** Kaggle GPU T4, Internet enabled, source `.mp4` attached under `/kaggle/input`. **Outputs:** `/kaggle/working/room_3d_handoff_v2`. Intermediate results are reused when available. Download the delivery ZIP and notebook before the Kaggle session ends.


## 1. Project configuration

A unique input video is selected automatically. For multiple attached MP4 files, set `VIDEO_OVERRIDE` to the full Kaggle input path. Project and pipeline directories are fixed for repeatable execution.


In [ ]:
from pathlib import Path
import subprocess
import sys
import json

PROJECT_DIR = Path("/kaggle/working/room_3d_handoff_v2")
PIPELINE_DIR = Path("/kaggle/working/person_a_pipeline")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_OVERRIDE = None
VIDEO_FILES = sorted(Path("/kaggle/input").rglob("*.mp4"))
preferred = [p for p in VIDEO_FILES if p.name.lower() == "meetingroom_3min.mp4"]
if VIDEO_OVERRIDE is not None:
    VIDEO_PATH = Path(VIDEO_OVERRIDE)
elif len(preferred) == 1:
    VIDEO_PATH = preferred[0]
elif len(VIDEO_FILES) == 1:
    VIDEO_PATH = VIDEO_FILES[0]
else:
    raise RuntimeError(f"Specify VIDEO_OVERRIDE. Available MP4 files: {VIDEO_FILES}")
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(VIDEO_PATH)
print(json.dumps({"video":str(VIDEO_PATH),"project_dir":str(PROJECT_DIR),"pipeline_dir":str(PIPELINE_DIR)},indent=2))


## 2. Independent stage scripts

All source files are declared below as standalone command-line stages. Notebook execution writes the authoritative scripts directly. No runtime source patches are required.

In [ ]:
%%writefile /kaggle/working/person_a_pipeline/common.py
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path


def project_root(value):
    path = Path(value).expanduser().resolve()
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')


def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def send_progress(args, stage, progress, status='running', error=None):
    event = {
        'stage': stage,
        'progress': int(progress),
        'status': status,
        'error': error,
        'timestamp_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()
    }
    root = project_root(args.project_dir)
    with (root / 'progress_events.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(event, ensure_ascii=False) + '\n')
    base = getattr(args, 'backend_url', None)
    task_id = getattr(args, 'task_id', None)
    if base and task_id:
        url = f"{base.rstrip('/')}/tasks/{task_id}/progress"
        headers = {'Content-Type': 'application/json'}
        token = os.environ.get('PERSON_A_BACKEND_TOKEN')
        if token:
            headers['Authorization'] = f'Bearer {token}'
        request = urllib.request.Request(url, data=json.dumps(event).encode('utf-8'), headers=headers, method='POST')
        try:
            with urllib.request.urlopen(request, timeout=10) as response:
                if response.status >= 400:
                    raise RuntimeError(f'HTTP {response.status}')
        except (urllib.error.URLError, TimeoutError, RuntimeError) as exc:
            print(f'Progress notification failed; retained locally: {exc}', flush=True)
    print(f'[{stage}] {progress}% {status}', flush=True)


def add_service_args(parser):
    parser.add_argument('--project_dir', required=True)
    parser.add_argument('--backend_url', default=None)
    parser.add_argument('--task_id', default=None)
    return parser


def elapsed(start):
    return round(time.perf_counter() - start, 3)


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/scene_data.py
import numpy as np
from pathlib import Path
from PIL import Image


def quat_rotation(q):
    w, x, y, z = np.asarray(q, dtype=np.float64) / np.linalg.norm(q)
    return np.array([[1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y)],[2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x)],[2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)]],dtype=np.float32)


def read_scene(dataset):
    dataset = Path(dataset)
    txt = dataset/'sparse_txt'
    cameras = {}
    for line in (txt/'cameras.txt').read_text().splitlines():
        if not line.strip() or line.startswith('#'):
            continue
        v = line.split()
        w,h = map(int, v[2:4])
        values = list(map(float,v[4:]))
        if v[1]=='SIMPLE_PINHOLE':
            f,cx,cy = values
            fx,fy = f,f
        elif v[1]=='PINHOLE':
            fx,fy,cx,cy = values
        else:
            raise RuntimeError(f'Image is not undistorted: {v[1]}')
        cameras[int(v[0])] = (w,h,np.array([[fx,0,cx],[0,fy,cy],[0,0,1]],dtype=np.float32))
    lines = [line.strip() for line in (txt/'images.txt').read_text().splitlines() if not line.startswith('#')]
    if len(lines)%2:
        raise RuntimeError('Invalid COLMAP images.txt pairs')
    records = []
    for header in lines[::2]:
        v = header.split()
        if len(v)<10:
            raise RuntimeError(f'Invalid COLMAP image: {header}')
        name = ' '.join(v[9:])
        width,height,K = cameras[int(v[8])]
        path = dataset/'images'/name
        if not path.is_file():
            raise FileNotFoundError(path)
        with Image.open(path) as image:
            if image.size!=(width,height):
                raise ValueError(f'Image and intrinsics dimensions do not match: {name}')
        w2c = np.eye(4,dtype=np.float32)
        w2c[:3,:3] = quat_rotation([float(x) for x in v[1:5]])
        w2c[:3,3] = [float(x) for x in v[5:8]]
        records.append({'name':name,'path':str(path),'width':width,'height':height,'viewmat':w2c,'K':K})
    records.sort(key=lambda item:item['name'])
    if len(records)<3:
        raise RuntimeError('Dataset has fewer than 3 cameras')
    points=[]
    colors=[]
    errors=[]
    for line in (txt/'points3D.txt').read_text().splitlines():
        if line.strip() and not line.startswith('#'):
            v=line.split()
            points.append([float(x) for x in v[1:4]])
            colors.append([float(x)/255 for x in v[4:7]])
            errors.append(float(v[7]))
    points=np.asarray(points,dtype=np.float32)
    colors=np.asarray(colors,dtype=np.float32)
    errors=np.asarray(errors,dtype=np.float32)
    if len(points)<50:
        raise RuntimeError('Not enough 3D seed points')
    return records,points,colors,errors


def holdout_indices(records,stride):
    if stride<3:
        raise ValueError('Holdout stride must be >=3')
    held = set(range(stride-1,len(records),stride))
    train=[i for i in range(len(records)) if i not in held]
    return train, sorted(held)


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/setup.py
import argparse
import importlib.util
import subprocess
import sys
import torch


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--vggt',action='store_true')
    parser.add_argument('--gsplat',action='store_true')
    args=parser.parse_args()
    dependencies=[('cv2','opencv-python-headless'),('PIL','Pillow'),('scipy','scipy'),('matplotlib','matplotlib'),('ninja','ninja')]
    if args.vggt:
        dependencies.extend([('einops','einops'),('huggingface_hub','huggingface_hub'),('safetensors','safetensors'),('omegaconf','omegaconf')])
    missing=[package for module,package in dependencies if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--no-input','-q',*missing],check=True)
    if args.gsplat and importlib.util.find_spec('gsplat') is None:
        subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','gsplat==1.5.3'],check=True)
    print({'torch_version':torch.__version__,'cuda_version':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'installed':missing})


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/extract.py
import argparse
import csv
import json
import shutil
import subprocess
import time
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from common import add_service_args, elapsed, project_root, save_json, send_progress


def execute(argv):
    result = subprocess.run(argv, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"{' '.join(map(str, argv))}\n{result.stderr[-3000:]}")
    return result.stdout


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--video', required=True)
    parser.add_argument('--fps', type=int, default=2)
    parser.add_argument('--long_side', type=int, default=960)
    parser.add_argument('--blur_threshold', type=float, default=60)
    parser.add_argument('--max_reject_percent', type=float, default=15)
    parser.add_argument('--full_fps', type=int, default=1)
    parser.add_argument('--benchmark_frames', type=int, default=32)
    parser.add_argument('--force', action='store_true')
    args = parser.parse_args()
    start = time.perf_counter()
    root = project_root(args.project_dir)
    source = Path(args.video)
    if not source.is_file():
        raise FileNotFoundError(source)
    if not 1 <= args.fps <= 4 or not 640 <= args.long_side <= 960:
        raise ValueError('fps must be 1–4 and long_side must be 640–960')
    if args.full_fps < 1 or args.fps % args.full_fps:
        raise ValueError('full_fps must divide fps')
    if not 0 <= args.max_reject_percent <= 50:
        raise ValueError('max_reject_percent must be 0–50')
    send_progress(args, 'extracting_frames', 5)
    probe = json.loads(execute(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=width,height,side_data_list:format=duration', '-of', 'json', str(source)]))
    raw_dir = root / 'frames_raw'
    clean_dir = root / 'frames_clean'
    full_dir = root / 'full_reconstruction' / 'images'
    benchmark_dir = root / 'pose_comparison' / 'images'
    qc_dir = root / 'qc'
    for directory in (raw_dir, clean_dir, full_dir, benchmark_dir, qc_dir):
        directory.mkdir(parents=True, exist_ok=True)
    if args.force or not list(raw_dir.glob('frame_*.jpg')):
        for image in raw_dir.glob('frame_*.jpg'):
            image.unlink()
        filter_graph = f'fps={args.fps},scale={args.long_side}:{args.long_side}:force_original_aspect_ratio=decrease:force_divisible_by=2:flags=lanczos'
        execute(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', str(source), '-vf', filter_graph, '-q:v', '2', '-start_number', '0', str(raw_dir / 'frame_%06d.jpg')])
    raw_paths = sorted(raw_dir.glob('frame_*.jpg'))
    if not raw_paths:
        raise RuntimeError('No frames extracted')
    image_size = Image.open(raw_paths[0]).size
    send_progress(args, 'filtering_frames', 15)
    scored = []
    for path in raw_paths:
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise RuntimeError(f'Unreadable frame: {path}')
        score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        scored.append((path, score))
    cutoff = min(args.blur_threshold, float(np.percentile([score for _, score in scored], args.max_reject_percent)))
    selected_clean = [p for p, score in scored if score >= cutoff]
    if args.force or set(p.name for p in clean_dir.glob('frame_*.jpg')) != set(p.name for p in selected_clean):
        for p in clean_dir.glob('frame_*.jpg'):
            p.unlink()
        for p in selected_clean:
            shutil.copy2(p, clean_dir / p.name)
    with (root / 'frame_quality.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['filename', 'laplacian_variance', 'kept'])
        for path, score in scored:
            writer.writerow([path.name, round(score, 4), int(score >= cutoff)])
    frame_names = sorted(p.name for p in selected_clean)
    stride = args.fps // args.full_fps
    selected_full = frame_names[::stride]
    if len(selected_full) < 3:
        raise RuntimeError('Insufficient frames to reconstruct')
    args.benchmark_frames = min(args.benchmark_frames, len(selected_full))
    middle = max(0, (len(selected_full) - args.benchmark_frames) // 2)
    selected_benchmark = selected_full[middle:middle + args.benchmark_frames]
    for directory, filenames in ((full_dir, selected_full), (benchmark_dir, selected_benchmark)):
        if args.force or {p.name for p in directory.glob('frame_*.jpg')} != set(filenames):
            for p in directory.glob('frame_*.jpg'):
                p.unlink()
            for filename in filenames:
                shutil.copy2(clean_dir / filename, directory / filename)
    for directory, filenames in ((full_dir.parent, selected_full), (benchmark_dir.parent, selected_benchmark)):
        manifest = [{'filename': name, 'approx_seconds': round(int(Path(name).stem.split('_')[-1]) / args.fps, 3)} for name in filenames]
        save_json(directory / 'dataset_manifest.json', {'source_video': source.name, 'fps': args.full_fps, 'count': len(filenames), 'image_resolution': list(image_size), 'frames': manifest})
    sample = [selected_clean[round(j*(len(selected_clean)-1)/15)] for j in range(16)]
    sheet = Image.new('RGB', (960, 640), 'white')
    for i, frame in enumerate(sample):
        with Image.open(frame) as im:
            thumb = ImageOps.fit(im.convert('RGB'), (240, 135))
        x, y = i % 4 * 240, i // 4 * 160
        sheet.paste(thumb, (x,y))
        ImageDraw.Draw(sheet).text((x+5,y+140), frame.name, fill='black')
    sheet.save(qc_dir / 'frames_preview.png')
    metadata = {'stage': 'extract', 'source_video': str(source), 'raw_frames': len(raw_paths), 'clean_frames': len(selected_clean), 'full_frames': len(selected_full), 'benchmark_frames': len(selected_benchmark), 'frames_resolution': list(image_size), 'fps_extraction': args.fps, 'fps_reconstruction': args.full_fps, 'blur_threshold': cutoff, 'duration_seconds': float(probe.get('format', {}).get('duration', 0)), 'processing_seconds': elapsed(start), 'rotation_handling': 'ffmpeg automatic rotation and aspect-ratio-preserving scale; dimensions read from extracted frames'}
    save_json(root / 'preprocessing_metadata.json', metadata)
    send_progress(args, 'extracting_frames', 25)
    print(json.dumps(metadata, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/install_colmap.py
import argparse
import json
import os
import subprocess
from pathlib import Path
from common import add_service_args, project_root, save_json


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--tools_dir', default='/kaggle/working/reconstruction_tools')
    args = parser.parse_args()
    root = project_root(args.project_dir)
    tools = Path(args.tools_dir)
    mamba = tools / 'bin' / 'micromamba'
    env_dir = tools / 'colmap_env'
    binary = env_dir / 'bin' / 'colmap'
    tools.mkdir(parents=True, exist_ok=True)
    if not mamba.exists():
        subprocess.run(['bash', '-lc', f'mkdir -p "{tools}" && curl -fLsS https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C "{tools}" bin/micromamba'], check=True)
    env = os.environ.copy()
    env['MAMBA_ROOT_PREFIX'] = str(tools / 'mamba_root')
    if not binary.is_file():
        subprocess.run([str(mamba), 'create', '-y', '-p', str(env_dir), '-c', 'conda-forge', '--override-channels', 'colmap=4.2.1'], env=env, check=True)
    help_result = subprocess.run([str(binary), '-h'], capture_output=True, text=True, env=env)
    if help_result.returncode or 'global_mapper' not in help_result.stdout + help_result.stderr:
        raise RuntimeError('COLMAP global_mapper missing: ' + (help_result.stdout + help_result.stderr)[-1000:])
    save_json(root / 'colmap_environment.json', {'colmap': str(binary), 'global_mapper_available': True, 'environment': str(env_dir), 'provider': 'conda-forge'})
    print(json.dumps({'colmap': str(binary), 'global_mapper_available': True}, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/poses.py
import argparse
import json
import os
import shutil
import sqlite3
import struct
import subprocess
import time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from common import add_service_args, elapsed, load_json, project_root, save_json, send_progress


def read_count(path):
    with Path(path).open('rb') as stream:
        return struct.unpack('<Q', stream.read(8))[0]


def run_command(binary, args, destination, env):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('w', encoding='utf-8') as stream:
        result = subprocess.run([str(binary)] + args, stdout=stream, stderr=subprocess.STDOUT, env=env)
    if result.returncode:
        raise RuntimeError(f'COLMAP failed: {destination}\n{destination.read_text()[-4000:]}')


def get_option(binary, command, candidates, env):
    result = subprocess.run([str(binary), command, '-h'], capture_output=True, text=True, env=env)
    help_text = result.stdout + result.stderr
    for name in candidates:
        if name in help_text:
            return name
    raise RuntimeError(f'No supported CPU option for {command}')


def plot_sparse(folder, text_dir):
    points = []
    for line in (text_dir / 'points3D.txt').read_text().splitlines():
        if line.strip() and not line.startswith('#'):
            words = line.split()
            points.append([float(v) for v in words[1:4]])
    centers = []
    for line in (text_dir / 'images.txt').read_text().splitlines():
        words = line.split()
        if len(words) < 10 or not words[9].lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        q = np.asarray([float(x) for x in words[1:5]], dtype=np.float64)
        w, x, y, z = q / np.linalg.norm(q)
        R = np.array([[1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y)],[2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x)],[2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)]])
        t = np.array([float(x) for x in words[5:8]])
        centers.append(-R.T @ t)
    if not points or not centers:
        return
    xyz = np.asarray(points)
    cs = np.asarray(centers)
    lo, hi = np.percentile(xyz, [1, 99], axis=0)
    xyz = xyz[np.all((xyz >= lo) & (xyz <= hi), axis=1)]
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], s=1, alpha=.55)
    ax.scatter(cs[:, 0], cs[:, 1], cs[:, 2], s=9, c='red')
    ax.set_title('COLMAP global mapping: sparse point cloud and cameras')
    ax.set_box_aspect(np.maximum(np.ptp(np.concatenate((xyz, cs)), axis=0), .001))
    fig.savefig(folder / 'sparse_preview.png', dpi=150)
    plt.close(fig)


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--mode', choices=['benchmark', 'full'], required=True)
    parser.add_argument('--colmap_bin', default='/kaggle/working/reconstruction_tools/colmap_env/bin/colmap')
    parser.add_argument('--camera_model', choices=['SIMPLE_PINHOLE', 'SIMPLE_RADIAL', 'OPENCV'], default='SIMPLE_RADIAL')
    parser.add_argument('--matching', choices=['sequential', 'exhaustive', 'both'], default='both')
    parser.add_argument('--reprocess', action='store_true')
    args = parser.parse_args()
    start = time.perf_counter()
    root = project_root(args.project_dir)
    parent = root / ('pose_comparison' if args.mode == 'benchmark' else 'full_reconstruction')
    images = parent / 'images'
    folder = parent / 'glomap'
    folder.mkdir(parents=True, exist_ok=True)
    binary = Path(args.colmap_bin)
    if not binary.is_file():
        raise FileNotFoundError(binary)
    filenames = sorted(p.name for p in images.glob('*.jpg'))
    if len(filenames) < 3:
        raise RuntimeError(f'Not enough images in {images}')
    meta_path = folder / 'mapping_metadata.json'
    sparse = folder / 'sparse' / '0'
    if meta_path.is_file() and (sparse / 'cameras.bin').is_file() and not args.reprocess:
        print(meta_path.read_text())
        send_progress(args, 'estimating_poses', 60 if args.mode == 'full' else 42)
        return
    env = os.environ.copy()
    env['QT_QPA_PLATFORM'] = 'offscreen'
    database = folder / 'database.db'
    if args.reprocess:
        for suffix in ('', '-wal', '-shm'):
            candidate = Path(str(database) + suffix)
            if candidate.exists():
                candidate.unlink()
    with sqlite3.connect(database) as con:
        has_images = con.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table' AND name='images'").fetchone()[0] > 0
        current_count = con.execute('SELECT COUNT(*) FROM images').fetchone()[0] if has_images else 0
        has_matches = con.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table' AND name='two_view_geometries'").fetchone()[0] > 0
        pair_count = con.execute('SELECT COUNT(*) FROM two_view_geometries WHERE rows>0').fetchone()[0] if has_matches else 0
    feature_seconds = matching_seconds = 0.
    if current_count != len(filenames) or pair_count == 0 or args.reprocess:
        for suffix in ('', '-shm', '-wal'):
            file = Path(str(database) + suffix)
            if file.exists():
                file.unlink()
        feature_option = get_option(binary, 'feature_extractor', ('--FeatureExtraction.use_gpu', '--SiftExtraction.use_gpu'), env)
        start_stage = time.perf_counter()
        run_command(binary, ['feature_extractor', '--database_path', str(database), '--image_path', str(images), '--ImageReader.single_camera', '1', '--ImageReader.camera_model', args.camera_model, feature_option, '0'], folder / 'feature_extraction.log', env)
        feature_seconds = elapsed(start_stage)
        send_progress(args, 'estimating_poses', 34 if args.mode == 'full' else 29)
        start_stage = time.perf_counter()
        if args.matching in ('sequential', 'both'):
            match_option = get_option(binary, 'sequential_matcher', ('--FeatureMatching.use_gpu', '--SiftMatching.use_gpu'), env)
            run_command(binary, ['sequential_matcher', '--database_path', str(database), '--SequentialMatching.overlap', '10', match_option, '0'], folder / 'sequential_matching.log', env)
        if args.matching in ('exhaustive', 'both'):
            match_option = get_option(binary, 'exhaustive_matcher', ('--FeatureMatching.use_gpu', '--SiftMatching.use_gpu'), env)
            run_command(binary, ['exhaustive_matcher', '--database_path', str(database), match_option, '0'], folder / 'exhaustive_matching.log', env)
        matching_seconds = elapsed(start_stage)
    with sqlite3.connect(database) as con:
        image_count = con.execute('SELECT COUNT(*) FROM images').fetchone()[0]
        feature_count = con.execute('SELECT COUNT(*) FROM keypoints WHERE rows>0').fetchone()[0]
        pairs = con.execute('SELECT COUNT(*) FROM two_view_geometries WHERE rows>0').fetchone()[0]
    if image_count != len(filenames) or feature_count < 3 or pairs == 0:
        raise RuntimeError('COLMAP database is incomplete')
    send_progress(args, 'estimating_poses', 45 if args.mode == 'full' else 35)
    candidate = folder / 'sparse_candidate'
    if candidate.exists():
        shutil.rmtree(candidate)
    candidate.mkdir()
    mapping_start = time.perf_counter()
    run_command(binary, ['global_mapper', '--database_path', str(database), '--image_path', str(images), '--output_path', str(candidate)], folder / 'global_mapping.log', env)
    mapping_seconds = elapsed(mapping_start)
    models = []
    for cameras_bin in candidate.rglob('cameras.bin'):
        directory = cameras_bin.parent
        if (directory/'images.bin').is_file() and (directory/'points3D.bin').is_file():
            models.append((int(read_count(directory/'images.bin')), int(read_count(directory/'points3D.bin')), directory))
    if not models:
        raise RuntimeError('No sparse model produced')
    registered, points, winner = max(models, key=lambda x: (x[0], x[1]))
    if registered < 2 or points < 1:
        raise RuntimeError('Insufficient registered cameras or sparse 3D points')
    final_sparse = folder / 'sparse'
    if final_sparse.exists():
        shutil.rmtree(final_sparse)
    final_sparse.mkdir()
    shutil.copytree(winner, final_sparse / '0', dirs_exist_ok=True)
    shutil.rmtree(candidate)
    text = folder / 'sparse_txt'
    if text.exists():
        shutil.rmtree(text)
    text.mkdir()
    run_command(binary, ['model_converter', '--input_path', str(final_sparse/'0'), '--output_path', str(text), '--output_type', 'TXT'], folder/'model_converter.log', env)
    plot_sparse(folder, text)
    info = {'stage': 'glomap_'+args.mode, 'input_images': len(filenames), 'registered_frames': registered, 'registration_ratio': round(registered/len(filenames),4), 'sparse_points': points, 'verified_pairs': pairs, 'feature_extraction_seconds': feature_seconds, 'matching_seconds': matching_seconds, 'global_mapping_seconds': mapping_seconds, 'total_seconds': elapsed(start), 'matching_strategy': args.matching, 'camera_model_used_for_feature_extraction': args.camera_model, 'gpu_sift': False, 'feature_matching_gpu_vram_gb': 0.0, 'global_mapper_peak_vram_gb': None, 'sparse_path': str(final_sparse/'0'), 'sparse_text_path': str(text), 'preview': str(folder/'sparse_preview.png'), 'notes': 'CPU SIFT/matching explicitly uses 0 GPU VRAM; global mapper GPU memory not instrumented. Re-run if comparing precise full-runtime metrics.'}
    save_json(meta_path, info)
    send_progress(args, 'estimating_poses', 60 if args.mode=='full' else 42)
    print(json.dumps(info, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/vggt_compare.py
import argparse
import gc
import json
import subprocess
import sys
import time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.spatial.transform import Rotation
from common import add_service_args, elapsed, project_root, save_json, send_progress


def align_similarity(source, target, src_R=None, dst_R=None):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    xm, ym = source.mean(0), target.mean(0)
    X, Y = source - xm, target - ym
    cov = Y.T @ X / max(len(X), 1)
    U, d, VT = np.linalg.svd(cov)
    D = np.diag([1., 1., np.linalg.det(U @ VT)])
    R = U @ D @ VT
    if np.linalg.matrix_rank(X) < 2 and src_R is not None and dst_R is not None:
        relative = dst_R @ np.transpose(src_R, (0, 2, 1))
        R = Rotation.from_matrix(relative).mean().as_matrix()
    variance = float(np.mean(np.sum(X * X, axis=1)))
    if variance < 1e-9:
        raise RuntimeError('Chunk overlap has negligible camera displacement; cannot estimate reliable scale')
    scale = float(np.sum((X @ R.T) * Y) / np.sum(X * X))
    if not np.isfinite(scale) or scale <= 1e-7:
        raise RuntimeError('Invalid VGGT chunk alignment scale')
    t = ym - scale * (R @ xm)
    residual = np.linalg.norm(scale * (source @ R.T) + t - target, axis=1)
    return scale, R, t, float(np.mean(residual))


def write_ply(path, points, colors):
    points = np.asarray(points, dtype=np.float32)
    colors = np.clip(np.asarray(colors) * 255., 0, 255).astype(np.uint8)
    dtype = np.dtype([('x', '<f4'), ('y', '<f4'), ('z', '<f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])
    array = np.empty(len(points), dtype=dtype)
    for j, key in enumerate(('x','y','z')):
        array[key] = points[:,j]
    for j, key in enumerate(('red','green','blue')):
        array[key] = colors[:,j]
    header = f'ply\nformat binary_little_endian 1.0\nelement vertex {len(points)}\nproperty float x\nproperty float y\nproperty float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n'
    with path.open('wb') as stream:
        stream.write(header.encode('ascii'))
        array.tofile(stream)


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--mode', required=True, choices=['benchmark','full'])
    parser.add_argument('--chunk_size', type=int, default=24)
    parser.add_argument('--overlap', type=int, default=8)
    parser.add_argument('--repository', default='/kaggle/working/vggt')
    parser.add_argument('--force', action='store_true')
    args = parser.parse_args()
    start = time.perf_counter()
    root = project_root(args.project_dir)
    parent = root / ('pose_comparison' if args.mode=='benchmark' else 'full_reconstruction')
    paths = sorted((parent / 'images').glob('*.jpg'))
    if len(paths) < 3:
        raise RuntimeError('Need at least three images for VGGT')
    if args.chunk_size <= args.overlap + 2 or args.overlap < 3:
        raise ValueError('chunk_size must exceed overlap+2, and overlap must be at least 3')
    output = parent / 'vggt'
    output.mkdir(parents=True, exist_ok=True)
    results_path = output / 'vggt_poses.npz'
    metadata_path = output / 'metadata.json'
    if results_path.exists() and metadata_path.exists() and not args.force:
        print(metadata_path.read_text())
        return
    if not torch.cuda.is_available():
        raise RuntimeError('VGGT needs CUDA')
    repo = Path(args.repository)
    if not (repo / 'vggt' / 'models' / 'vggt.py').is_file():
        subprocess.run(['git','clone','https://github.com/facebookresearch/vggt.git',str(repo)],check=True)
    subprocess.run(['git','-C',str(repo),'checkout','a288dd0f14786c93483e45524328726ab7b1b4ce'],check=True)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    send_progress(args,'estimating_poses_vggt',30)
    model_load_start = time.perf_counter()
    model = VGGT.from_pretrained('facebook/VGGT-1B').to('cuda:0').eval()
    load_seconds = elapsed(model_load_start)
    all_Rcw = [None] * len(paths)
    all_C = [None] * len(paths)
    all_K = [None] * len(paths)
    residuals = []
    sampled_points = []
    sampled_colors = []
    chunk_seconds = []
    torch.cuda.set_device('cuda:0')
    step = args.chunk_size - args.overlap
    begins = list(range(0, len(paths), step)) if args.mode=='full' else [0]
    for chunk_index, begin in enumerate(begins):
        end = min(begin + (args.chunk_size if args.mode=='full' else len(paths)), len(paths))
        indices = list(range(begin, end))
        if not indices:
            continue
        if args.mode=='full' and end - begin < args.overlap + 1 and chunk_index:
            indices = list(range(max(0, len(paths)-args.chunk_size), len(paths)))
            if all_C[indices[-1]] is not None:
                break
        batch_start = time.perf_counter()
        batch = load_and_preprocess_images([str(paths[j]) for j in indices]).to('cuda:0')
        with torch.inference_mode():
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                prediction = model(batch)
            E,K = pose_encoding_to_extri_intri(prediction['pose_enc'], batch.shape[-2:])
        torch.cuda.synchronize()
        e = E.squeeze(0).float().cpu().numpy()
        k = K.squeeze(0).float().cpu().numpy()
        local_Rcw = np.transpose(e[:,:,:3], (0,2,1))
        local_C = -np.einsum('nij,nj->ni', local_Rcw, e[:,:,3])
        overlap_local = [j for j, global_index in enumerate(indices) if all_C[global_index] is not None]
        if chunk_index == 0:
            scale, global_rot, global_t, error = 1., np.eye(3), np.zeros(3), 0.
        elif len(overlap_local) >= 3:
            scale,global_rot,global_t,error = align_similarity(
                local_C[overlap_local],np.asarray([all_C[indices[j]] for j in overlap_local]),
                local_Rcw[overlap_local],np.asarray([all_Rcw[indices[j]] for j in overlap_local])
            )
        else:
            raise RuntimeError(f'Insufficient overlap in VGGT chunk starting at frame {begin}')
        residuals.append(error)
        global_C = scale * (local_C @ global_rot.T) + global_t
        global_Rcw = global_rot @ local_Rcw
        for local_index, global_index in enumerate(indices):
            if all_C[global_index] is None:
                all_C[global_index] = global_C[local_index]
                all_Rcw[global_index] = global_Rcw[local_index]
                all_K[global_index] = k[local_index]
        points_tensor = prediction.get('world_points')
        if points_tensor is not None:
            world = points_tensor.detach().float().cpu().numpy().squeeze(0)
            rgb_images = batch.detach().float().cpu().numpy()
            if world.ndim == 4 and world.shape[1:3] == tuple(batch.shape[-2:]):
                for local_index, global_index in enumerate(indices):
                    if global_index not in indices[:args.overlap] or chunk_index == 0:
                        p = world[local_index][::12,::12].reshape(-1,3)
                        colors = np.moveaxis(rgb_images[local_index],0,-1)[::12,::12].reshape(-1,3)
                        good = np.isfinite(p).all(axis=1)
                        sampled_points.append((scale * (p[good] @ global_rot.T) + global_t).astype(np.float32))
                        sampled_colors.append(np.clip(colors[good],0,1).astype(np.float32))
        chunk_seconds.append(elapsed(batch_start))
        print(f'VGGT chunk {chunk_index+1}: {begin}..{end-1}, residual={error:.5f}, seconds={chunk_seconds[-1]:.2f}', flush=True)
        del prediction,batch,E,K
        gc.collect()
        torch.cuda.empty_cache()
        if args.mode == 'benchmark':
            break
    if any(x is None for x in all_C):
        raise RuntimeError(f'VGGT chunks covered only {sum(x is not None for x in all_C)}/{len(paths)} frames')
    C = np.asarray(all_C, dtype=np.float32)
    Rcw = np.asarray(all_Rcw, dtype=np.float32)
    Rc = np.transpose(Rcw, (0,2,1))
    extr = np.concatenate((Rc,-np.einsum('nij,nj->ni',Rc,C)[:,:,None]),axis=2)
    intr = np.asarray(all_K, dtype=np.float32)
    np.savez_compressed(results_path, extrinsics=extr, intrinsics=intr, image_names=np.array([x.name for x in paths]))
    plt.figure(figsize=(8,6))
    ax = plt.axes(projection='3d')
    ax.plot(C[:,0],C[:,1],C[:,2],'-o',markersize=2)
    if sampled_points:
        cloud = np.concatenate(sampled_points,axis=0)
        colors = np.concatenate(sampled_colors,axis=0)
        write_ply(output/'vggt_pointcloud_preview.ply', cloud,colors)
        preview = cloud[::max(1,len(cloud)//20000)]
        ax.scatter(preview[:,0],preview[:,1],preview[:,2],s=.1,alpha=.15)
    ax.set_title(f'VGGT {args.mode}: chunk-aligned camera trajectory / points')
    plt.savefig(output/'vggt_pose_pointcloud.png',dpi=170)
    plt.close()
    info = {'stage': 'vggt_'+args.mode, 'frames': len(paths), 'predicted_poses': len(C), 'geometrically_verified_registered_frames': None, 'input_frame_size': 'VGGT internally resized; check model_image_hw', 'model_image_hw': list(rgb_images.shape[-2:]) if sampled_points else None, 'batch_chunk_size': args.chunk_size if args.mode=='full' else len(paths), 'overlap': args.overlap if args.mode=='full' else 0, 'alignment_residual_mean': float(np.mean(residuals)), 'alignment_residual_max': float(np.max(residuals)), 'inference_chunk_seconds': round(sum(chunk_seconds),2), 'model_loading_seconds': load_seconds, 'total_seconds': elapsed(start), 'peak_gpu_allocated_vram_gb': round(torch.cuda.max_memory_allocated()/1024**3,3), 'pose_path': str(results_path), 'point_cloud_preview': str(output/'vggt_pointcloud_preview.ply') if sampled_points else None, 'notes': 'Similarity-aligned overlapping VGGT chunks are research estimates; validate for drift, scale consistency and point-cloud quality. Camera predictions are not equivalent to SfM registration.'}
    save_json(metadata_path, info)
    send_progress(args,'estimating_poses_vggt',42 if args.mode=='benchmark' else 62)
    print(json.dumps(info,indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/prepare_dataset.py
import argparse
import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from PIL import Image
from common import add_service_args, elapsed, project_root, save_json, send_progress


def run(argv, log):
    env = os.environ.copy()
    env['QT_QPA_PLATFORM'] = 'offscreen'
    with log.open('w') as stream:
        result = subprocess.run(argv, stdout=stream, stderr=subprocess.STDOUT, env=env)
    if result.returncode:
        raise RuntimeError(f'COLMAP failed: {log}\n{log.read_text()[-4000:]}')


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--colmap_bin', default='/kaggle/working/reconstruction_tools/colmap_env/bin/colmap')
    parser.add_argument('--force', action='store_true')
    args = parser.parse_args()
    start = time.perf_counter()
    root = project_root(args.project_dir)
    model = root/'full_reconstruction'/'glomap'/'sparse'/'0'
    source_images = root/'full_reconstruction'/'images'
    dataset = root/'3dgs_dataset'
    if not (model/'cameras.bin').is_file() or not list(source_images.glob('*.jpg')):
        raise RuntimeError('Full GLOMAP reconstruction is missing')
    if (dataset/'sparse'/'0'/'cameras.bin').exists() and not args.force:
        print((dataset/'metadata.json').read_text())
        return
    if dataset.exists() and args.force:
        shutil.rmtree(dataset)
    candidate = root/'3dgs_dataset_candidate'
    if candidate.exists():
        shutil.rmtree(candidate)
    candidate.mkdir()
    send_progress(args,'preparing_training_data',63)
    bin_path = Path(args.colmap_bin)
    if not bin_path.exists():
        raise FileNotFoundError(bin_path)
    run([str(bin_path),'image_undistorter','--image_path',str(source_images),'--input_path',str(model),'--output_path',str(candidate),'--output_type','COLMAP'],root/'undistort.log')
    images_dir = candidate/'images'
    sparse_dir = candidate/'sparse'
    if not images_dir.is_dir() or not sparse_dir.is_dir():
        raise RuntimeError('Undistortion did not produce expected COLMAP dataset')
    if (sparse_dir/'cameras.bin').exists():
        sparse_dir.rename(candidate/'sparse_direct')
        (candidate/'sparse').mkdir()
        (candidate/'sparse_direct').rename(candidate/'sparse'/'0')
    elif not (sparse_dir/'0'/'cameras.bin').exists():
        raise RuntimeError('Undistorted sparse cameras are missing')
    txt = candidate/'sparse_txt'
    txt.mkdir()
    run([str(bin_path),'model_converter','--input_path',str(candidate/'sparse'/'0'),'--output_path',str(txt),'--output_type','TXT'],root/'undistorted_model_converter.log')
    camera_lines = [x.split() for x in (txt/'cameras.txt').read_text().splitlines() if x.strip() and not x.startswith('#')]
    if not camera_lines or any(x[1] not in ('PINHOLE','SIMPLE_PINHOLE') for x in camera_lines):
        raise RuntimeError('Image undistorter did not yield pinhole camera models')
    count = len(list(images_dir.glob('*.jpg')))
    meta = {'stage':'dataset_preparation','image_count':count,'original_camera_model':'see full_reconstruction/glomap/sparse_txt/cameras.txt','undistorted_camera_models':sorted({x[1] for x in camera_lines}),'image_dimensions':list(Image.open(next(images_dir.glob('*.jpg'))).size),'seconds':elapsed(start),'undistortion':'COLMAP image_undistorter; use these transformed intrinsics and images together'}
    save_json(candidate/'metadata.json',meta)
    if dataset.exists():
        shutil.rmtree(dataset)
    candidate.rename(dataset)
    send_progress(args,'preparing_training_data',65)
    print(json.dumps(meta, indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/train.py
import argparse
import csv
import json
import math
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.spatial import cKDTree
from gsplat.rendering import rasterization
from gsplat.strategy import DefaultStrategy
from scene_data import read_scene, holdout_indices
from common import add_service_args, elapsed, project_root, save_json, send_progress


def ssim(a,b):
    a = a.permute(2,0,1).unsqueeze(0)
    b = b.permute(2,0,1).unsqueeze(0)
    ma, mb = F.avg_pool2d(a, 11, stride=1, padding=5), F.avg_pool2d(b, 11, stride=1, padding=5)
    va = F.avg_pool2d(a*a,11,stride=1,padding=5)-ma*ma
    vb = F.avg_pool2d(b*b,11,stride=1,padding=5)-mb*mb
    covariance = F.avg_pool2d(a*b,11,stride=1,padding=5)-ma*mb
    C1,C2 = .01**2,.03**2
    value = ((2*ma*mb+C1)*(2*covariance+C2))/((ma*ma+mb*mb+C1)*(va+vb+C2)).clamp_min(1e-8)
    return value.clamp(-1,1).mean()


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--steps', type=int, default=7000)
    parser.add_argument('--schedule_steps', type=int, default=15000)
    parser.add_argument('--run_name', default='stable_7000')
    parser.add_argument('--holdout_stride', type=int, default=10)
    parser.add_argument('--sh_degree', type=int, default=2, choices=[1,2,3])
    parser.add_argument('--ssim_weight', type=float, default=0.0)
    parser.add_argument('--scale_regularization', type=float, default=0.0)
    parser.add_argument('--scale_cap_fraction', type=float, default=0.20)
    parser.add_argument('--lr_final_factor', type=float, default=0.1)
    parser.add_argument('--save_every', type=int, default=500)
    parser.add_argument('--validate_every', type=int, default=500)
    parser.add_argument('--refine_stop_iter', type=int, default=0)
    parser.add_argument('--resume', action='store_true')
    args = parser.parse_args()
    if args.steps<1 or args.schedule_steps<args.steps or not 0<=args.ssim_weight<=1:
        raise ValueError('Invalid step count or SSIM weight')
    root = project_root(args.project_dir)
    records,xyz,rgb,errors = read_scene(root/'3dgs_dataset')
    train_ids,val_ids = holdout_indices(records,args.holdout_stride)
    if not val_ids:
        raise RuntimeError('No held-out frames for evaluation')
    run = root/'3dgs_training'/args.run_name
    run.mkdir(parents=True,exist_ok=True)
    latest, best = run/'latest.pt',run/'best.pt'
    device='cuda:0'
    if not torch.cuda.is_available():
        raise RuntimeError('Enable Kaggle Tesla T4')
    keep = np.isfinite(xyz).all(axis=1)&np.isfinite(rgb).all(axis=1)&np.isfinite(errors)
    keep &= errors<=np.percentile(errors[keep],99) if keep.any() else keep
    xyz,rgb=xyz[keep],rgb[keep]
    distances = cKDTree(xyz).query(xyz,k=2)[0][:,1]
    positive=distances[(distances>0)&np.isfinite(distances)]
    if len(positive)<3:
        raise RuntimeError('Seed point distances are invalid')
    lo,hi=np.percentile(positive,[10,90])
    init_scales=np.log(np.clip(distances,lo,hi)*0.75).astype(np.float32)
    sh_dim=(args.sh_degree+1)**2
    if args.resume and latest.exists():
        state=torch.load(latest,map_location=device,weights_only=False)
        if state['holdout_stride'] != args.holdout_stride or state['sh_degree'] != args.sh_degree or state['schedule_steps'] != args.schedule_steps:
            raise RuntimeError('Cannot resume with different split/SH/scheduler; choose a new run_name')
        splats={name:torch.nn.Parameter(value.detach().to(device)) for name,value in state['splats'].items()}
        start_step=int(state['step'])
    else:
        if latest.exists():
            raise RuntimeError('Run exists. Supply --resume or a different --run_name')
        splats={
            'means':torch.nn.Parameter(torch.tensor(xyz,device=device)),
            'quats':torch.nn.Parameter(torch.tensor([[1.,0.,0.,0.]],device=device).repeat(len(xyz),1)),
            'scales':torch.nn.Parameter(torch.tensor(init_scales[:,None].repeat(3,axis=1),device=device)),
            'opacities':torch.nn.Parameter(torch.full((len(xyz),),-1.5,device=device)),
            'sh0':torch.nn.Parameter(torch.tensor(((rgb-.5)/0.28209479177387814)[:,None,:],device=device)),
            'shN':torch.nn.Parameter(torch.zeros((len(xyz),sh_dim-1,3),device=device))
        }
        start_step=0
        state=None
    rates={'means':1e-4,'quats':1e-3,'scales':3e-3,'opacities':2e-2,'sh0':1e-2,'shN':5e-4}
    opts={name:torch.optim.Adam([{'params':[value],'lr':rates[name]}],eps=1e-15) for name,value in splats.items()}
    strategy=DefaultStrategy(refine_start_iter=500,refine_stop_iter=args.refine_stop_iter,refine_every=100,reset_every=3000,verbose=False)
    strategy.check_sanity(splats,opts)
    vmat=torch.tensor(np.stack([x['viewmat'] for x in records]),device=device)
    K=torch.tensor(np.stack([x['K'] for x in records]),device=device)
    centers=torch.linalg.inv(vmat)[:,:3,3]
    scene_scale=max(float(torch.linalg.norm(centers-centers.mean(0),dim=-1).max().item()),.1)
    rates.update({'means':1.6e-4*scene_scale,'quats':1e-3,'scales':5e-3,'opacities':5e-2,'sh0':2.5e-3,'shN':1.25e-4})
    strategy_state=strategy.initialize_state(scene_scale=scene_scale)
    if state is not None:
        for name,opt in opts.items():
            opt.load_state_dict(state['optimizers'][name])
        strategy_state=state['strategy_state']
        best_score=float(state.get('best_psnr',float('-inf')))
    else:
        best_score=float('-inf')
    image_cache=[]
    for r in records:
        with Image.open(r['path']) as im:
            image_cache.append(torch.from_numpy(np.array(im.convert('RGB'),dtype=np.uint8)))
    width,height=records[0]['width'],records[0]['height']
    if any(x['width']!=width or x['height']!=height for x in records):
        raise ValueError('Mixed image dimensions are not supported')
    background=torch.tensor([.5,.5,.5],device=device)
    def render(index,grad=False):
        return rasterization(
            means=splats['means'],quats=F.normalize(splats['quats'],dim=-1),
            scales=torch.exp(splats['scales'].clamp(-12,3)),opacities=torch.sigmoid(splats['opacities']),
            colors=torch.cat([splats['sh0'],splats['shN']],dim=1),sh_degree=args.sh_degree,
            viewmats=vmat[index:index+1],Ks=K[index:index+1],width=width,height=height,
            backgrounds=background,packed=True,absgrad=grad and strategy.absgrad
        )
    def validate():
        indices=np.linspace(0,len(val_ids)-1,min(len(val_ids),8)).round().astype(int)
        selected=[val_ids[i] for i in sorted(set(indices))]
        scores=[]
        with torch.no_grad():
            for frame in selected:
                prediction,_,_=render(frame)
                truth=image_cache[frame].to(device,dtype=torch.float32)/255
                mse=F.mse_loss(prediction[0].clamp(0,1),truth).clamp_min(1e-10)
                scores.append(float((-10*torch.log10(mse)).item()))
        return float(np.mean(scores))
    def training_psnr():
        scores=[]
        with torch.no_grad():
            for frame in train_ids:
                prediction,_,_=render(frame)
                truth=image_cache[frame].to(device,dtype=torch.float32)/255
                mse=F.mse_loss(prediction[0].clamp(0,1),truth).clamp_min(1e-10)
                scores.append(float((-10*torch.log10(mse)).item()))
        return float(np.mean(scores))
    def checkpoint(path,step,best_psnr):
        temporary=path.with_suffix('.tmp')
        torch.save({'step':step,'splats':{k:v.detach().cpu() for k,v in splats.items()},'optimizers':{k:o.state_dict() for k,o in opts.items()},'strategy_state':strategy_state,'sh_degree':args.sh_degree,'schedule_steps':args.schedule_steps,'holdout_stride':args.holdout_stride,'best_psnr':best_psnr,'train_count':len(train_ids),'heldout_count':len(val_ids)},temporary)
        temporary.replace(path)
    monitor_positions = np.linspace(0,len(val_ids)-1,min(8,len(val_ids))).round().astype(int)
    monitor_ids = sorted({val_ids[int(i)] for i in monitor_positions})
    test_ids = [i for i in val_ids if i not in set(monitor_ids)]
    train_list_path=run/'split.json'
    save_json(train_list_path,{'trained_frames':[records[i]['name'] for i in train_ids],'heldout_frames':[records[i]['name'] for i in val_ids], 'monitor_frames':[records[i]['name'] for i in monitor_ids], 'test_frames':[records[i]['name'] for i in test_ids],'note':'Held-out images are excluded from splat optimization but were used in SfM pose reconstruction.'})
    rng=np.random.default_rng(2026)
    train_order=np.concatenate([
        rng.permutation(train_ids)
        for _ in range((args.steps+len(train_ids)-1)//len(train_ids))
    ])
    history=run/'history.csv'
    history_exists=history.exists()
    torch.cuda.set_device('cuda:0')
    start=time.perf_counter()
    send_progress(args,'training_gaussians',65)
    if start_step>=args.steps:
        print(f'Already completed {start_step} steps')
        return
    with history.open('a',newline='') as stream:
        writer=csv.writer(stream)
        if not history_exists:
            writer.writerow(['step','train_l1','train_ssim','total_loss','gaussians','heldout_sample_psnr_db','lr_multiplier','train_mean_psnr_db'])
        for step in range(start_step+1,args.steps+1):
            frame=int(train_order[step-1])
            target=image_cache[frame].to(device,dtype=torch.float32)/255.
            lr_factor=args.lr_final_factor**(step/args.schedule_steps)
            for name,opt in opts.items():
                opt.param_groups[0]['lr']=rates[name]*lr_factor
                opt.zero_grad(set_to_none=True)
            rendered,alphas,info=render(frame,True)
            strategy.step_pre_backward(splats,opts,strategy_state,step,info)
            prediction=rendered[0]
            l1=F.l1_loss(prediction,target)
            structural=ssim(prediction,target)
            spread=torch.exp(splats['scales'].clamp(-12,3)).max(dim=1).values
            penalty=torch.relu(spread/(scene_scale*args.scale_cap_fraction)-1).square().mean()
            loss=(1-args.ssim_weight)*l1+args.ssim_weight*(1-structural)+args.scale_regularization*penalty
            if not torch.isfinite(loss):
                raise RuntimeError(f'Nonfinite loss at step {step}')
            loss.backward()
            for opt in opts.values():
                opt.step()
            strategy.step_post_backward(splats,opts,strategy_state,step,info,packed=True)
            if step%args.validate_every==0 or step==args.steps:
                heldout_psnr=validate()
                train_psnr=training_psnr()
                writer.writerow([step,round(l1.item(),6),round(structural.item(),6),round(loss.item(),6),len(splats['means']),round(heldout_psnr,3),round(lr_factor,6),round(train_psnr,3)])
                stream.flush()
                print(f'Step {step}/{args.steps} | gaussians={len(splats["means"])} | train PSNR={train_psnr:.3f} dB | heldout PSNR={heldout_psnr:.3f} dB',flush=True)
                if heldout_psnr>best_score:
                    best_score=heldout_psnr
                    checkpoint(best,step,best_score)
            if step%args.save_every==0 or step==args.steps:
                checkpoint(latest,step,best_score)
                send_progress(args,'training_gaussians',65+int(25*step/args.steps))
    torch.cuda.synchronize()
    meta={'stage':'gaussian_training','run':args.run_name,'steps_completed':args.steps,'starting_step':start_step,'time_this_run_seconds':elapsed(start),'cumulative_training_seconds':round(elapsed(start)+(json.loads((run/'training_metadata.json').read_text()).get('cumulative_training_seconds',0) if (run/'training_metadata.json').exists() else 0),3),'gaussians':len(splats['means']),'peak_allocated_gpu_vram_gb':round(torch.cuda.max_memory_allocated()/1024**3,3),'sh_degree':args.sh_degree,'loss':'(1-ssim_weight)*L1 + ssim_weight*(1-SSIM) + scale_regularization*penalty','heldout_frames':len(val_ids),'training_frames':len(train_ids),'best_heldout_sample_psnr_db':round(best_score,3),'best_checkpoint':str(best),'latest_checkpoint':str(latest),'lr_schedule_steps':args.schedule_steps,'note':'Fixed seeded training schedule, L1 with unclipped loss, and no densification by default. Monitoring views are separate from final test views; COLMAP SfM used all images.'}
    save_json(run/'training_metadata.json',meta)
    send_progress(args,'training_gaussians',90)
    print(json.dumps(meta,indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/evaluate.py
import argparse
import csv
import time
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from gsplat.rendering import rasterization
from scene_data import holdout_indices,read_scene
from train import ssim
from common import add_service_args, project_root, save_json


def main():
    parser=add_service_args(argparse.ArgumentParser())
    parser.add_argument('--run_name',default='stable_7000')
    parser.add_argument('--checkpoint',choices=['best','latest'],default='latest')
    args=parser.parse_args()
    start=time.perf_counter()
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU is unavailable. Enable a Kaggle T4 accelerator before evaluation.')
    torch.cuda.set_device('cuda:0')
    root=project_root(args.project_dir)
    folder=root/'3dgs_training'/args.run_name
    path=folder/f'{args.checkpoint}.pt'
    if not path.exists():
        raise FileNotFoundError(path)
    state=torch.load(path,map_location='cuda:0',weights_only=False)
    records,_,_,_=read_scene(root/'3dgs_dataset')
    train_ids,heldout_ids=holdout_indices(records,state['holdout_stride'])
    monitor_positions=np.linspace(0,len(heldout_ids)-1,min(8,len(heldout_ids))).round().astype(int)
    monitor_set={heldout_ids[int(i)] for i in monitor_positions}
    test_set=set(heldout_ids)-monitor_set
    splats=state['splats']
    shdegree=int(state['sh_degree'])
    device='cuda:0'
    means=splats['means'].float().to(device)
    quats=F.normalize(splats['quats'].float().to(device),dim=-1)
    scales=torch.exp(splats['scales'].float().to(device).clamp(-12,3))
    opacities=torch.sigmoid(splats['opacities'].float().to(device))
    colors=torch.cat([splats['sh0'],splats['shN']],dim=1).float().to(device)
    mats=torch.tensor(np.stack([r['viewmat'] for r in records]),device=device)
    Ks=torch.tensor(np.stack([r['K'] for r in records]),device=device)
    background=torch.tensor([.5,.5,.5],device=device)
    rows=[]
    saved_panels=[]
    with torch.inference_mode():
        for index in heldout_ids:
            rec=records[index]
            with Image.open(rec['path']) as im:
                target=torch.tensor(np.array(im.convert('RGB'),dtype=np.uint8),device=device).float()/255
            prediction,alpha,_=rasterization(means=means,quats=quats,scales=scales,opacities=opacities,colors=colors,sh_degree=shdegree,viewmats=mats[index:index+1],Ks=Ks[index:index+1],width=rec['width'],height=rec['height'],backgrounds=background,packed=True)
            pred=prediction[0].clamp(0,1)
            mse=F.mse_loss(pred,target).clamp_min(1e-10)
            psnr=(-10*torch.log10(mse)).item()
            similarity=ssim(pred,target).item()
            rows.append({'image':rec['name'],'split':'monitor' if index in monitor_set else 'test','psnr_db':round(psnr,4),'ssim':round(similarity,5),'l1':round(F.l1_loss(pred,target).item(),5),'mean_alpha':round(alpha.mean().item(),5)})
            if len(saved_panels)<6:
                saved_panels.append((rec['name'],np.array((target.cpu().numpy()*255).astype(np.uint8)),np.array((pred.cpu().numpy()*255).astype(np.uint8))))
    report=folder/'evaluation'
    report.mkdir(parents=True,exist_ok=True)
    with (report/'heldout_metrics.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    if saved_panels:
        canvas=Image.new('RGB',(2*480,len(saved_panels)*300),'white')
        draw=ImageDraw.Draw(canvas)
        for row,(name,gt,pred) in enumerate(saved_panels):
            yy=row*300
            draw.text((8,yy+4),f'{name} | heldout original',fill='black')
            draw.text((488,yy+4),f'{name} | prediction',fill='black')
            canvas.paste(Image.fromarray(gt).resize((480,270)),(0,yy+30))
            canvas.paste(Image.fromarray(pred).resize((480,270)),(480,yy+30))
        canvas.save(report/'heldout_contact_sheet.jpg',quality=85)
    test_rows=[row for row in rows if row['split']=='test']
    monitor_rows=[row for row in rows if row['split']=='monitor']
    summary={'test_frames':len(test_rows),'monitor_frames':len(monitor_rows),'test_mean_psnr_db':round(float(np.mean([r['psnr_db'] for r in test_rows])),3) if test_rows else None,'test_mean_ssim':round(float(np.mean([r['ssim'] for r in test_rows])),4) if test_rows else None,'monitor_mean_psnr_db':round(float(np.mean([r['psnr_db'] for r in monitor_rows])),3) if monitor_rows else None,'stage':'heldout_evaluation','run':args.run_name,'evaluated_frames':len(rows),'training_frames':len(train_ids),'mean_psnr_db':round(float(np.mean([r['psnr_db'] for r in rows])),3),'mean_ssim':round(float(np.mean([r['ssim'] for r in rows])),4),'mean_l1':round(float(np.mean([r['l1'] for r in rows])),5),'worst_5':sorted(rows,key=lambda r:r['psnr_db'])[:5],'checkpoint':str(path),'split_note':'Heldout images were not used for GS optimization but did participate in SfM camera registration; free-camera generalization requires visual review.'}
    summary['runtime_seconds']=round(time.perf_counter()-start,3)
    summary['peak_allocated_gpu_vram_gb']=round(torch.cuda.max_memory_allocated()/1024**3,3)
    split_file=folder/'split.json'
    split_data=json.loads(split_file.read_text()) if split_file.is_file() else {}
    split_data['trained_frames']=[records[i]['name'] for i in train_ids]
    split_data['heldout_frames']=[records[i]['name'] for i in heldout_ids]
    split_data['monitor_frames']=[records[i]['name'] for i in heldout_ids if i in monitor_set]
    split_data['test_frames']=[records[i]['name'] for i in heldout_ids if i in test_set]
    save_json(split_file,split_data)
    save_json(report/'heldout_summary.json',summary)
    print(json.dumps(summary,indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/export.py
import argparse
import json
import time
from pathlib import Path
import numpy as np
import torch
from common import add_service_args, project_root, save_json,send_progress


def sigmoid(x):
    return 1/(1+np.exp(-np.clip(x,-30,30)))


def main():
    parser=add_service_args(argparse.ArgumentParser())
    parser.add_argument('--run_name',default='stable_7000')
    parser.add_argument('--checkpoint',choices=['best','latest'],default='latest')
    parser.add_argument('--checkpoint_path',default=None)
    args=parser.parse_args()
    start=time.perf_counter()
    root=project_root(args.project_dir)
    path=Path(args.checkpoint_path) if args.checkpoint_path else root/'3dgs_training'/args.run_name/f'{args.checkpoint}.pt'
    if not path.is_file():
        raise FileNotFoundError(path)
    state=torch.load(path,map_location='cpu',weights_only=False)
    p=state.get('splats',state.get('parameters'))
    def array(key):
        return p[key].detach().float().cpu().numpy().astype(np.float32)
    is_new='sh0' in p
    means=array('means')
    log_scales=np.clip(array('scales' if is_new else 'scales'),-12,3)
    q=array('quats')
    q_norm=np.linalg.norm(q,axis=1,keepdims=True)
    if np.any(q_norm<1e-6):
        raise RuntimeError('Invalid rotations')
    q=q/q_norm
    opacity=array('opacities')
    if is_new:
        sh0=array('sh0')[:,0,:]
        shn=array('shN')
        degree=int(state['sh_degree'])
    else:
        rgb=sigmoid(array('colors'))
        sh0=(rgb-.5)/.28209479177387814
        shn=np.empty((len(means),0,3),dtype=np.float32)
        degree=0
    rgb=np.clip(.5+sh0*.28209479177387814,0,1).astype(np.float32)
    alpha=sigmoid(opacity).astype(np.float32)
    scales=np.exp(log_scales).astype(np.float32)
    n=len(means)
    if not all(len(x)==n for x in (log_scales,q,opacity,sh0,alpha)):
        raise ValueError('Parameter count mismatch')
    if not all(np.isfinite(x).all() for x in (means,log_scales,q,opacity,sh0,shn)):
        raise RuntimeError('Nonfinite gaussian values')
    rest=shn.transpose(0,2,1).reshape(n,-1)
    fields=['x','y','z','nx','ny','nz']+[f'f_dc_{j}' for j in range(3)]+[f'f_rest_{j}' for j in range(rest.shape[1])]+['opacity']+[f'scale_{j}' for j in range(3)]+[f'rot_{j}' for j in range(4)]
    dtype=np.dtype([(name,'<f4') for name in fields])
    vertices=np.zeros(n,dtype=dtype)
    for j,key in enumerate(('x','y','z')):
        vertices[key]=means[:,j]
    for j in range(3):
        vertices[f'f_dc_{j}']=sh0[:,j]
        vertices[f'scale_{j}']=log_scales[:,j]
    for j in range(rest.shape[1]):
        vertices[f'f_rest_{j}']=rest[:,j]
    for j in range(4):
        vertices[f'rot_{j}']=q[:,j]
    vertices['opacity']=opacity
    export_dir=root/'exports'/args.run_name
    export_dir.mkdir(parents=True,exist_ok=True)
    header='\n'.join(['ply','format binary_little_endian 1.0',f'element vertex {n}']+[f'property float {key}' for key in fields]+['end_header'])+'\n'
    ply=export_dir/'scene.ply'
    with ply.open('wb') as stream:
        stream.write(header.encode('ascii'))
        vertices.tofile(stream)
    splat_dtype=np.dtype([('pos','<f4',(3,)),('scales','<f4',(3,)),('color','u1',(4,)),('quat','u1',(4,))])
    if splat_dtype.itemsize!=32:
        raise RuntimeError('Unexpected SPLAT byte layout')
    order=np.argsort(alpha*np.prod(scales,axis=1))[::-1]
    data=np.empty(n,dtype=splat_dtype)
    data['pos']=means[order]
    data['scales']=scales[order]
    rgba=np.concatenate((rgb,alpha[:,None]),axis=1)
    data['color']=np.clip(np.round(rgba[order]*255),0,255).astype('u1')
    data['quat']=np.clip(np.round(128+128*q[order]),0,255).astype('u1')
    splat=export_dir/'scene.splat'
    data.tofile(splat)
    if ply.stat().st_size!=len(header.encode('ascii'))+n*dtype.itemsize or splat.stat().st_size!=n*32:
        raise RuntimeError('Export size validation failed')
    record={'stage':'export','checkpoint_step':int(state['step']),'checkpoint_path':str(path),'gaussian_count':n,'sh_degree':degree,'ply':str(ply),'splat':str(splat),'ply_size_mb':round(ply.stat().st_size/1024**2,2),'splat_size_mb':round(splat.stat().st_size/1024**2,2),'notes':'PLY preserves SH coefficients; legacy SPLAT bakes only SH DC and needs viewer compatibility testing. Camera orientation is COLMAP convention, so set the viewer transform explicitly.'}
    record['runtime_seconds']=round(time.perf_counter()-start,3)
    save_json(export_dir/'scene_metadata.json',record)
    send_progress(args,'exporting',96)
    print(json.dumps(record,indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/package.py
import argparse
import json
import zipfile
from pathlib import Path
import torch
from common import add_service_args, project_root, save_json, send_progress


def main():
    parser=add_service_args(argparse.ArgumentParser())
    parser.add_argument('--scripts_dir',default=None)
    parser.add_argument('--notebook_path',default=None)
    parser.add_argument('--run_name',default='stable_7000')
    parser.add_argument('--include_video',action='store_true')
    args=parser.parse_args()
    root=project_root(args.project_dir)
    run=root/'3dgs_training'/args.run_name
    checkpoint=run/'latest.pt'
    scene=root/'exports'/args.run_name/'scene.ply'
    if not checkpoint.is_file() or not scene.is_file():
        raise RuntimeError('Training checkpoint and exported scene are required before packaging')
    required=[
        root/'3dgs_dataset/images',
        root/'3dgs_dataset/sparse_txt/cameras.txt',
        root/'3dgs_dataset/sparse_txt/images.txt',
        root/'3dgs_dataset/sparse_txt/points3D.txt',
        root/'full_reconstruction/images',
        root/'exports'/args.run_name/'scene.splat'
    ]
    missing=[str(item) for item in required if not item.exists()]
    if missing:
        raise FileNotFoundError('Missing required handoff artifacts: '+str(missing))
    weights_path=run/'inference_weights.pt'
    state=torch.load(checkpoint,map_location='cpu',weights_only=False)
    torch.save({'step':state['step'],'splats':{name:value.detach().cpu() for name,value in state['splats'].items()},'sh_degree':state['sh_degree'],'holdout_stride':state['holdout_stride'],'schedule_steps':state['schedule_steps']},weights_path)
    fixed=[
        'preprocessing_metadata.json','frame_quality.csv','colmap_environment.json',
        'pose_comparison/dataset_manifest.json','full_reconstruction/dataset_manifest.json',
        'pose_comparison/vggt/metadata.json','pose_comparison/vggt/vggt_poses.npz',
        'pose_comparison/vggt/vggt_pose_pointcloud.png','pose_comparison/vggt/vggt_pointcloud_preview.ply',
        'pose_comparison/glomap/mapping_metadata.json','pose_comparison/glomap/sparse_preview.png',
        'full_reconstruction/vggt/metadata.json','full_reconstruction/vggt/vggt_poses.npz',
        'full_reconstruction/vggt/vggt_pose_pointcloud.png','full_reconstruction/vggt/vggt_pointcloud_preview.ply',
        'full_reconstruction/glomap/mapping_metadata.json','full_reconstruction/glomap/sparse_preview.png',
        '3dgs_dataset/metadata.json',
        f'3dgs_training/{args.run_name}/training_metadata.json',
        f'3dgs_training/{args.run_name}/history.csv',
        f'3dgs_training/{args.run_name}/split.json',
        f'3dgs_training/{args.run_name}/inference_weights.pt',
        f'3dgs_training/{args.run_name}/evaluation/heldout_metrics.csv',
        f'3dgs_training/{args.run_name}/evaluation/heldout_summary.json',
        f'3dgs_training/{args.run_name}/evaluation/heldout_contact_sheet.jpg',
        f'exports/{args.run_name}/scene.ply',f'exports/{args.run_name}/scene.splat',
        f'exports/{args.run_name}/scene_metadata.json',
        'reports/comparison.md','reports/runtime_vram.csv','reports/week2_handoff.md',
        'reports/validation_status.json','qc/frames_preview.png'
    ]
    folders=[
        'full_reconstruction/images',
        '3dgs_dataset/images','3dgs_dataset/sparse/0','3dgs_dataset/sparse_txt',
        'full_reconstruction/glomap/sparse/0','full_reconstruction/glomap/sparse_txt',
        'pose_comparison/glomap/sparse/0','pose_comparison/glomap/sparse_txt'
    ]
    files={root/item for item in fixed if (root/item).is_file()}
    for folder in folders:
        base=root/folder
        if base.is_dir():
            files.update(x for x in base.rglob('*') if x.is_file())
    script_dir=Path(args.scripts_dir) if args.scripts_dir else Path(__file__).resolve().parent
    output=root/'Person_A_backend_delivery.zip'
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for file in sorted(files):
            archive.write(file,arcname=str(Path('artifacts')/file.relative_to(root)))
        for script in sorted(script_dir.glob('*.py')):
            archive.write(script,arcname=f'pipeline/{script.name}')
        if args.notebook_path and Path(args.notebook_path).is_file():
            archive.write(args.notebook_path,arcname='notebooks/Person_A_Reconstruction.ipynb')
        archive.writestr('README.txt','Person A ML reconstruction package. CLI entrypoint: pipeline/run_pipeline.py. Undistorted photographs: artifacts/3dgs_dataset/images. Pinhole intrinsics and world-to-camera extrinsics: artifacts/3dgs_dataset/sparse_txt. Exported scene: artifacts/exports/'+args.run_name+'/scene.ply and scene.splat. Splat colors contain SH DC only, whereas PLY preserves higher SH terms. Inference weights do not contain optimizer state. Full resume checkpoint remains in the Kaggle project folder and is not transferred. Open artifacts/reports/week2_handoff.md for camera conventions and integration paths.\n')
        if args.include_video:
            for candidate in sorted(root.glob('*.mp4')):
                archive.write(candidate,arcname='source/'+candidate.name)
    record={'stage':'backend_delivery','zip_path':str(output),'zip_size_mb':round(output.stat().st_size/1024**2,2),'files':len(files),'scripts':len(list(script_dir.glob('*.py'))),'run_name':args.run_name,'contains_optimizer_state':False,'contains_clean_images':(root/'3dgs_dataset/images').is_dir(),'contains_colmap_poses':(root/'3dgs_dataset/sparse_txt/images.txt').is_file()}
    save_json(root/'delivery_manifest.json',record)
    send_progress(args,'packaging',100,status='completed')
    print(json.dumps(record,indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/run_pipeline.py
import argparse
import json
import subprocess
import sys
from pathlib import Path


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--project_dir',required=True)
    parser.add_argument('--video',default=None)
    parser.add_argument('--stage',choices=['extract','install_colmap','vggt_benchmark','vggt_full','glomap_benchmark','glomap_full','prepare','train','evaluate','export','report','package','production','all'],required=True)
    parser.add_argument('--steps',type=int,default=7000)
    parser.add_argument('--run_name',default='stable_7000')
    parser.add_argument('--backend_url',default=None)
    parser.add_argument('--task_id',default=None)
    parser.add_argument('--resume',action='store_true')
    parser.add_argument('--matching',choices=['sequential','both','exhaustive'],default=None)
    parser.add_argument('--colmap_bin',default='/kaggle/working/reconstruction_tools/colmap_env/bin/colmap')
    args=parser.parse_args()
    script_dir=Path(__file__).parent
    common=['--project_dir',args.project_dir]
    if args.backend_url:
        common+=['--backend_url',args.backend_url]
    if args.task_id:
        common+=['--task_id',args.task_id]
    commands={
        'extract':['extract.py','--video',args.video] if args.video else None,
        'install_colmap':['install_colmap.py'],
        'vggt_benchmark':['vggt_compare.py','--mode','benchmark'],
        'vggt_full':['vggt_compare.py','--mode','full'],
        'glomap_benchmark':['poses.py','--mode','benchmark','--colmap_bin',args.colmap_bin],
        'glomap_full':['poses.py','--mode','full','--colmap_bin',args.colmap_bin,'--matching',args.matching or ('sequential' if args.stage=='production' else 'both')],
        'prepare':['prepare_dataset.py','--colmap_bin',args.colmap_bin],
        'train':['train.py','--steps',str(args.steps),'--schedule_steps','15000','--run_name',args.run_name,'--refine_stop_iter','0','--ssim_weight','0','--scale_regularization','0']+(['--resume'] if args.resume else []),
        'evaluate':['evaluate.py','--run_name',args.run_name,'--checkpoint','latest'],
        'export':['export.py','--run_name',args.run_name,'--checkpoint','latest'],
        'package':['package.py','--scripts_dir',str(script_dir),'--run_name',args.run_name],
        'report':['report.py','--run_name',args.run_name]
    }
    if args.stage=='production':
        stages=['extract','install_colmap','glomap_full','prepare','train','evaluate','export','report','package']
    elif args.stage=='all':
        stages=['extract','install_colmap','vggt_benchmark','glomap_benchmark','vggt_full','glomap_full','prepare','train','evaluate','export','report','package']
    else:
        stages=[args.stage]
    for stage in stages:
        if stage=='train':
            checkpoint=Path(args.project_dir)/'3dgs_training'/args.run_name/'latest.pt'
            if checkpoint.is_file():
                import torch
                saved=torch.load(checkpoint,map_location='cpu',weights_only=False)
                if int(saved['step'])>=args.steps:
                    print(f'SKIP train: checkpoint already at step {saved["step"]}',flush=True)
                    continue
                if not args.resume:
                    raise RuntimeError('Training checkpoint exists; use --resume to continue')
        item=commands[stage]
        if item is None:
            raise ValueError(f'{stage} requires --video')
        command=[sys.executable,str(script_dir/item[0]),*common,*item[1:]]
        print(f'RUNNING {stage}: {command}',flush=True)
        subprocess.run(command,check=True)
    print(json.dumps({'completed_stages':stages,'project_dir':args.project_dir},indent=2))


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/person_a_pipeline/report.py
import argparse
import csv
import json
import math
from common import add_service_args, project_root, save_json


def main():
    parser = add_service_args(argparse.ArgumentParser())
    parser.add_argument('--run_name', default='stable_7000')
    args = parser.parse_args()
    root = project_root(args.project_dir)
    reports = root / 'reports'
    reports.mkdir(parents=True, exist_ok=True)

    def get(name):
        p = root / name
        return json.loads(p.read_text()) if p.is_file() else {}

    def fmt(val):
        return 'not measured' if val is None else str(val)

    pre = get('preprocessing_metadata.json')
    vb = get('pose_comparison/vggt/metadata.json')
    vf = get('full_reconstruction/vggt/metadata.json')
    gb = get('pose_comparison/glomap/mapping_metadata.json')
    gf = get('full_reconstruction/glomap/mapping_metadata.json')
    und = get('3dgs_dataset/metadata.json')
    train = get(f'3dgs_training/{args.run_name}/training_metadata.json')
    ev = get(f'3dgs_training/{args.run_name}/evaluation/heldout_summary.json')
    ex = get(f'exports/{args.run_name}/scene_metadata.json')

    rows = [
        ['Extraction and filtering', pre.get('processing_seconds'), 0, 'CPU decode and filtering'],
        ['VGGT 32-frame benchmark', vb.get('total_seconds'), vb.get('peak_gpu_allocated_vram_gb'), 'Includes loading; PyTorch allocated peak'],
        ['GLOMAP 32-frame benchmark', gb.get('total_seconds'), 0, 'CPU SIFT/matching 0 GB; global mapping GPU peak unmeasured'],
        ['VGGT full chunk merge', vf.get('total_seconds'), vf.get('peak_gpu_allocated_vram_gb'), 'Includes model loading; PyTorch allocated peak'],
        ['GLOMAP full reconstruction', gf.get('total_seconds'), 0, 'CPU SIFT/matching 0 GB; global mapping GPU peak unmeasured'],
        ['COLMAP image undistortion', und.get('seconds'), 0, 'CPU stage'],
        ['Gaussian training', train.get('cumulative_training_seconds', train.get('time_this_run_seconds')), train.get('peak_allocated_gpu_vram_gb'), 'PyTorch allocated peak; excludes JIT compilation'],
        ['Held-out evaluation', ev.get('runtime_seconds'), ev.get('peak_allocated_gpu_vram_gb'), 'Unmeasured values left blank'],
        ['Gaussian PLY/SPLAT export', ex.get('runtime_seconds'), 0, 'CPU model export']
    ]
    with (reports / 'runtime_vram.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['stage', 'runtime_seconds', 'allocated_gpu_vram_gb', 'measurement_notes'])
        writer.writerows(rows)

    chunk_count = len(vf.get('chunk_seconds', [])) if vf.get('chunk_seconds') else None
    if chunk_count is None and vf.get('frames') and vf.get('batch_chunk_size') and vf.get('overlap'):
        chunk_count = 1 + math.ceil(max(0, vf['frames'] - vf['batch_chunk_size']) / (vf['batch_chunk_size'] - vf['overlap']))

    comparison = f'''# VGGT vs GLOMAP — Reconstruction Benchmark

VGGT and GLOMAP were evaluated on matching frame identities from the same video. VGGT uses internally resized images and predicts poses without SfM registration; GLOMAP uses extracted frames and verifies image registration geometrically.

| Metric | VGGT benchmark | GLOMAP benchmark | VGGT full sequence | GLOMAP full sequence |
|---|---:|---:|---:|---:|
| Input frames | {fmt(vb.get('frames'))} | {fmt(gb.get('input_images'))} | {fmt(vf.get('frames'))} | {fmt(gf.get('input_images'))} |
| Predicted / registered frames | {fmt(vb.get('predicted_poses'))} predicted | {fmt(gb.get('registered_frames'))} registered | {fmt(vf.get('predicted_poses'))} predicted | {fmt(gf.get('registered_frames'))} registered |
| Sparse triangulated points | N/A (dense predicted cloud) | {fmt(gb.get('sparse_points'))} | N/A (dense predicted cloud) | {fmt(gf.get('sparse_points'))} |
| Runtime (seconds) | {fmt(vb.get('total_seconds'))} | {fmt(gb.get('total_seconds'))} | {fmt(vf.get('total_seconds'))} | {fmt(gf.get('total_seconds'))} |
| GPU VRAM (GB) | {fmt(vb.get('peak_gpu_allocated_vram_gb'))} allocated | 0 CPU matching; mapper unmeasured | {fmt(vf.get('peak_gpu_allocated_vram_gb'))} allocated | 0 CPU matching; mapper unmeasured |
| Resolution | {fmt(vb.get('model_image_hw'))} H×W | Extracted frames | {fmt(vf.get('model_image_hw'))} H×W | Extracted frames |
| Relative verification | Chunk residual not applicable | {fmt(gb.get('verified_pairs'))} verified pairs | {fmt(vf.get('alignment_residual_mean'))} mean chunk-alignment residual | {fmt(gf.get('verified_pairs'))} verified pairs |

**VGGT chunking:** {fmt(chunk_count)} overlapping chunks were used on the full sequence. The alignment residual measures internal agreement between overlapping chunks; it is not a ground-truth absolute trajectory error.

**Selected method:** GLOMAP supplies geometrically registered cameras, COLMAP intrinsics, and sparse geometry suitable for undistortion and Gaussian training. VGGT is substantially faster on the full sequence but its camera estimates and merged point map require additional validation before using them for reconstruction. The full GLOMAP run used {fmt(gf.get('matching_strategy'))} feature matching and spent {fmt(gf.get('matching_seconds'))} seconds matching. Production defaults to sequential matching for lower CPU cost, with less loop-closure coverage as a trade-off.

**Measurement boundary:** 0 GB refers to CPU SIFT feature extraction and matching. GLOMAP global mapper peak GPU allocation was not independently instrumented and is not reported as a measured zero. The methods also use different effective image resolutions, so the runtimes are not resolution-controlled.

**Visual checks:** `pose_comparison/vggt/vggt_pose_pointcloud.png`, `pose_comparison/glomap/sparse_preview.png`, `full_reconstruction/vggt/vggt_pose_pointcloud.png`, `full_reconstruction/glomap/sparse_preview.png`.
'''
    (reports / 'comparison.md').write_text(comparison)

    handoff = f'''# Week 2 — Backend and Viewer Handoff

**Project root:** `{root}`

| Artifact | Relative path |
|---|---|
| Full video frames | `full_reconstruction/images/` |
| Undistorted frames | `3dgs_dataset/images/` |
| Undistorted pinhole intrinsics | `3dgs_dataset/sparse_txt/cameras.txt` |
| COLMAP registered poses | `3dgs_dataset/sparse_txt/images.txt` |
| Sparse 3D points | `3dgs_dataset/sparse_txt/points3D.txt` |
| Binary sparse model | `3dgs_dataset/sparse/0/` |
| VGGT full trajectory and internal intrinsics | `full_reconstruction/vggt/vggt_poses.npz` |
| SH degree {fmt(ex.get('sh_degree'))} Gaussian model | `exports/{args.run_name}/scene.ply` |
| Compact DC-only-color scene | `exports/{args.run_name}/scene.splat` |
| Inference-only Gaussian weights | `3dgs_training/{args.run_name}/inference_weights.pt` |
| Train/monitor/test frame split | `3dgs_training/{args.run_name}/split.json` |
| Held-out metrics and images | `3dgs_training/{args.run_name}/evaluation/` |
| Individual backend-ready stages | `pipeline/*.py` in the ZIP |

**Camera coordinates.** COLMAP uses local camera +X right, +Y down, +Z forward. The stored quaternion and translation are world-to-camera. World-space camera centers are `-R.T @ t`. For an OpenGL-style camera-to-world view, use `T_gl = inv(T_colmap_world_to_camera) @ diag(1,-1,-1,1)`. The explicit camera-axis correction is a 180-degree rotation around local X: `[[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]]`. This camera-axis correction does not itself require rotating the scene points. If a viewer needs an additional world transform, apply the same chosen transform to both scene Gaussians and camera-to-world matrices. No unique scene-world flip was calibrated in SuperSplat; import compatibility was visually checked but the initial view was not aligned against an exact source pose.

**Intrinsics integrity.** Undistorted images from `image_undistorter` must be paired with the resulting pinhole calibration. Do not apply the undistorted camera intrinsics to pre-undistortion frames. The PLY contains higher-degree spherical harmonic coefficients while the legacy `.splat` stores only DC colors. Viewer rendering and the initial camera must be validated independently.

**Backend interface.** Stage scripts accept `--project_dir`. Optional `--backend_url` and `--task_id` enable `POST /tasks/{{id}}/progress`; events are also logged locally in `progress_events.jsonl`. Work is invoked by subprocesses; scheduling, API authentication, and persistent task queuing belong to the backend worker implementation. Persist the ZIP in backend storage before Kaggle sessions expire.

**Evaluation.** The model reached {fmt(ex.get('checkpoint_step'))} iterations with {fmt(ex.get('gaussian_count'))} Gaussians. Gaussian optimization excluded {fmt(ev.get('evaluated_frames'))} held-out frames, with {fmt(ev.get('test_frames'))} test frames not used for best-checkpoint selection; all frames participated in the SfM solution. Mean PSNR across all held-out views: {fmt(ev.get('mean_psnr_db'))} dB; separately monitored-test PSNR: {fmt(ev.get('test_mean_psnr_db'))} dB. These view-interpolation metrics do not prove unrestricted novel-view quality. A longer 15,000-iteration run must be separately recorded if performed.
'''
    (reports / 'week2_handoff.md').write_text(handoff)
    viewer_path=reports/'viewer_validation.json'
    viewer_record=json.loads(viewer_path.read_text()) if viewer_path.is_file() else {}
    status = {
        'run_name': args.run_name,
        'training_steps': train.get('steps_completed'),
        'gaussians': ex.get('gaussian_count'),
        'heldout_mean_psnr_db': ev.get('mean_psnr_db'),
        'heldout_mean_ssim': ev.get('mean_ssim'),
        'test_mean_psnr_db': ev.get('test_mean_psnr_db'),
        'test_mean_ssim': ev.get('test_mean_ssim'),
        'viewer_note': viewer_record.get('notes','Viewer quality not recorded for this project'),
        'completed_15000_steps': bool(ex.get('checkpoint_step', 0) >= 15000),
        'test_scope': 'Test frames withheld from 3DGS training and checkpoint selection, but not from SfM'
    }
    save_json(reports / 'validation_status.json', status)
    print(json.dumps(status, indent=2))


if __name__ == '__main__':
    main()


## 3. Dependencies

Install the Python libraries while preserving the PyTorch/CUDA version supplied by Kaggle. COLMAP is installed separately into an isolated micromamba environment. The gsplat CUDA kernel may compile at first use; subsequent launches can reuse the local JIT cache.


In [ ]:
subprocess.run([sys.executable, str(PIPELINE_DIR / "setup.py"), "--vggt", "--gsplat"], check=True)


## 4. Frame extraction and quality filtering

Default extraction is 2 FPS, with 1 FPS images selected for the full sparse reconstruction and a 32-frame controlled benchmark. Extracted image dimensions are read from saved frames, so rotated inputs maintain their actual aspect ratio.


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "extract.py"),"--project_dir",str(PROJECT_DIR),"--video",str(VIDEO_PATH),"--fps","2","--full_fps","1","--long_side","960","--benchmark_frames","32"],check=True)


## 5. COLMAP global-mapper environment

COLMAP 4.2.1 is provisioned once and reused on subsequent runs. The `global_mapper` command provides the GLOMAP-style global reconstruction output.


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "install_colmap.py"),"--project_dir",str(PROJECT_DIR)],check=True)


## 6. VGGT and GLOMAP benchmark on identical 32 frames

VGGT estimates camera poses and dense point maps in a single forward pass. COLMAP estimates feature geometry and uses a global mapper to register cameras. VGGT internally resizes inputs, while COLMAP works at the extracted resolution; the report records this methodological limitation.


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "vggt_compare.py"),"--project_dir",str(PROJECT_DIR),"--mode","benchmark"],check=True)


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "poses.py"),"--project_dir",str(PROJECT_DIR),"--mode","benchmark","--camera_model","SIMPLE_RADIAL","--matching","both"],check=True)


## 7. Full-video VGGT chunk alignment

Process the complete selected frame sequence in overlapping 24-frame chunks with an 8-frame overlap. Export merged poses and a point-cloud/trajectory preview. Sim(3) alignment residuals measure internal chunk consistency, not external pose accuracy.


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "vggt_compare.py"),"--project_dir",str(PROJECT_DIR),"--mode","full","--chunk_size","24","--overlap","8"],check=True)


## 8. Full GLOMAP reconstruction and camera undistortion

Register the full 1-FPS image set with a distortion-aware camera model. Sequential and exhaustive matching provide loop-closure candidates at additional CPU cost. `image_undistorter` produces consistent pinhole images and intrinsics for 3DGS.


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "poses.py"),"--project_dir",str(PROJECT_DIR),"--mode","full","--camera_model","SIMPLE_RADIAL","--matching","both"],check=True)


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "prepare_dataset.py"),"--project_dir",str(PROJECT_DIR)],check=True)


## 9. Reconstruction visual quality check

Compare the estimated point clouds and camera trajectories before Gaussian optimization. Sparse geometry and camera registration are complementary indicators and must be checked alongside photometric quality.


In [ ]:
from PIL import Image
from IPython.display import display

previews = [
    PROJECT_DIR / "qc/frames_preview.png",
    PROJECT_DIR / "pose_comparison/vggt/vggt_pose_pointcloud.png",
    PROJECT_DIR / "pose_comparison/glomap/sparse_preview.png",
    PROJECT_DIR / "full_reconstruction/vggt/vggt_pose_pointcloud.png",
    PROJECT_DIR / "full_reconstruction/glomap/sparse_preview.png"
]
for image_path in previews:
    print(image_path.relative_to(PROJECT_DIR))
    if image_path.is_file():
        with Image.open(image_path) as preview:
            display(preview.copy())
    else:
        print("Preview unavailable")


## 10. Stable Gaussian Splatting training — 7,000 iterations

Training uses the camera intrinsics and undistorted images produced by COLMAP. The validated configuration uses SH degree 2, unclipped render output for L1 optimization, scene-scaled position learning rate, seeded frame permutations, learning-rate decay and disabled adaptive densification. SSIM and scale regularization remain supported as command-line options but are set to zero for this validated run. Training and checkpoint monitoring scores are recorded independently from the final test subset.

The run writes checkpoints under `3dgs_training/stable_7000/`. If the latest saved checkpoint already reached step 7,000, the stage is skipped rather than rerun.


For the validated 7,000-step run, SSIM and densification are disabled. The failed densification experiment should be retained only as a research diagnostic, not as the selected production model.

In [ ]:
import torch

RUN_NAME = "stable_7000"
CHECKPOINT_PATH = PROJECT_DIR / "3dgs_training" / RUN_NAME / "latest.pt"
completed_steps = 0
if CHECKPOINT_PATH.is_file():
    completed_steps = int(torch.load(CHECKPOINT_PATH,map_location="cpu",weights_only=False)["step"])
if completed_steps < 7000:
    command = [sys.executable,str(PIPELINE_DIR / "train.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME,"--steps","7000","--schedule_steps","15000","--sh_degree","2","--refine_stop_iter","0","--ssim_weight","0","--scale_regularization","0","--validate_every","500","--save_every","1000"]
    if CHECKPOINT_PATH.is_file():
        command.append("--resume")
    subprocess.run(command,check=True)
else:
    print(f"Existing training checkpoint: step {completed_steps}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_path = PROJECT_DIR / "3dgs_training" / RUN_NAME / "history.csv"
if not history_path.is_file():
    raise FileNotFoundError(history_path)
history = pd.read_csv(history_path)
if history.empty:
    raise RuntimeError("Training history is empty")
display(history.tail(8))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history["step"], history["train_mean_psnr_db"], label="Train PSNR")
ax.plot(history["step"], history["heldout_sample_psnr_db"], label="Monitor PSNR")
ax.set_xlabel("Training step")
ax.set_ylabel("PSNR (dB)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 11. Full held-out evaluation and SH2 scene export

Use `latest.pt` to avoid selecting an export using the same held-out evaluation views. The held-out pool is separated into checkpoint-monitoring and test subsets for transparent reporting. All images contributed to the original SfM reconstruction, so this is not an entirely independent free-camera test.


In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "evaluate.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME,"--checkpoint","latest"],check=True)
subprocess.run([sys.executable,str(PIPELINE_DIR / "export.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME,"--checkpoint","latest"],check=True)

summary = json.loads((PROJECT_DIR / "3dgs_training" / RUN_NAME / "evaluation/heldout_summary.json").read_text())
scene = json.loads((PROJECT_DIR / "exports" / RUN_NAME / "scene_metadata.json").read_text())
print(json.dumps({"step":scene["checkpoint_step"],"gaussians":scene["gaussian_count"],"heldout_psnr_db":summary["mean_psnr_db"],"test_psnr_db":summary.get("test_mean_psnr_db"),"heldout_ssim":summary["mean_ssim"],"ply":scene["ply"],"splat":scene["splat"]},indent=2))
with Image.open(PROJECT_DIR / "3dgs_training" / RUN_NAME / "evaluation/heldout_contact_sheet.jpg") as preview:
    display(preview.copy())


## 12. Optional continuation to 15,000 iterations

For longer training, change `RUN_15000_EXTENSION` to `True`. The 15,000-step schedule resumes from `latest.pt` and keeps the stable training configuration. Re-run evaluation, export and reporting after continuation. The validated 7,000-step artifact is the default reproducible result; an extension must be evaluated before replacing it in a production viewer.


In [ ]:
RUN_15000_EXTENSION = False
if RUN_15000_EXTENSION:
    subprocess.run([sys.executable,str(PIPELINE_DIR / "train.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME,"--steps","15000","--schedule_steps","15000","--sh_degree","2","--refine_stop_iter","0","--ssim_weight","0","--scale_regularization","0","--validate_every","500","--save_every","1000","--resume"],check=True)
    subprocess.run([sys.executable,str(PIPELINE_DIR / "evaluate.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME,"--checkpoint","latest"],check=True)
    subprocess.run([sys.executable,str(PIPELINE_DIR / "export.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME,"--checkpoint","latest"],check=True)


## 13. Benchmark comparison, runtime/VRAM, and Week 2 handoff

Generate reports from recorded stage metadata, including the full 180-frame comparison. GLOMAP CPU matching uses 0 GPU VRAM; the mapper peak is uninstrumented. A viewer note is included only if recorded in `reports/viewer_validation.json`.

In [ ]:
from IPython.display import Markdown

subprocess.run([sys.executable,str(PIPELINE_DIR / "report.py"),"--project_dir",str(PROJECT_DIR),"--run_name",RUN_NAME],check=True)
for report_name in ["comparison.md","week2_handoff.md"]:
    display(Markdown((PROJECT_DIR / "reports" / report_name).read_text()))
import pandas as pd
display(pd.read_csv(PROJECT_DIR / "reports/runtime_vram.csv"))



## 14. Backend delivery package

Package the selected GLOMAP model, undistorted frames and calibration, VGGT research artifacts, SH2 PLY, DC-only SPLAT, inference weights and reports. The archive intentionally excludes optimizer states, experiments, raw extraction duplicates, and the full matching database. Transfer the ZIP to backend storage before the Kaggle session ends.

In [ ]:
subprocess.run([sys.executable,str(PIPELINE_DIR / "package.py"),"--project_dir",str(PROJECT_DIR),"--scripts_dir",str(PIPELINE_DIR),"--run_name",RUN_NAME],check=True)
print("Backend delivery:",PROJECT_DIR / "Person_A_backend_delivery.zip")


## Delivery status

The reconstruction and inference artifacts are packaged from recorded output files. The selected run is `stable_7000` with adaptive densification disabled. The original unstable densification run is not a production result. The PLY includes SH2 coefficients; the compact SPLAT stores DC colors only and may render differently. The 15,000-step continuation and production worker deployment remain optional tasks and must not be reported as completed unless executed.